In [26]:
import pandas as pd
import numpy as np
import json
import requests
import io
import time
import os
from dotenv import load_dotenv

In [27]:
load_dotenv()
API_KEY = os.getenv("API_KEY")

In order to construct the series IDs for Local Area Unemployment Statistics (LAUS), Current Employment Statistics (CES), or Consumer Price Index (CPI), the query string must include FIPS (for state and area) NAICS (for industry) codes. This can be done manually, but I found it's easiest access the downloads resource on the BLS website:

https://download.bls.gov/pub/time.series/

These files can be read directly from the server so long as a user agent is specified.

In [ ]:
headers = {'User-Agent': 'Economic Data Acquisition Project for STAT 386/0.0 (n1ckaust@byu.edu)'}

def get_bls_metadata(url):
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return pd.read_csv(io.StringIO(response.text), sep="\t")

state_fips_df = get_bls_metadata("https://download.bls.gov/pub/time.series/sm/sm.state")
industry_df = get_bls_metadata("https://download.bls.gov/pub/time.series/sm/sm.industry")
area_cu_df = get_bls_metadata("https://download.bls.gov/pub/time.series/cu/cu.area")


In [ ]:
print(state_fips_df.head())
# print(area_fips_df.head())
print(industry_df.head())
print(area_cu_df.head())

   state_code  state_name
0           0  All States
1           1     Alabama
2           2      Alaska
3           4     Arizona
4           5    Arkansas
   industry_code              industry_name
0              0              Total Nonfarm
1        5000000              Total Private
2        6000000            Goods Producing
3        7000000          Service-Providing
4        8000000  Private Service Providing
  area_code          area_name  display_level selectable  sort_sequence
0      0000  U.S. city average              0          T              1
1      0100          Northeast              0          T              5
2      0110        New England              1          T             10
3      0120    Middle Atlantic              1          T             11
4      0200            Midwest              0          T             14


In [42]:
state_fips_df["state_code"] = state_fips_df["state_code"].astype(str).str.zfill(2)
industry_df["industry_code"] = industry_df["industry_code"].astype(str).str.zfill(8)
area_cu_df["area_code"] = area_cu_df["area_code"].astype(str).str.zfill(4)

There are more industries than these, but these are the largest supersectors, and are more representative.

In [ ]:
industry_keep = [
    "00000000",  # Total Nonfarm
    "05000000",  # Total Private
    "06000000",  # Goods Producing
    "07000000",  # Service-Providing
    "08000000",  # Private Service Providing
    "20000000",  # Construction
    "30000000",  # Manufacturing
    "40000000",  # Trade, Transportation, and Utilities
    "50000000",  # Information 
    "55000000",  # Financial Activities
    "60000000",  # Professional and Business Services
    "65000000",  # Education and Health Services
    "70000000",  # Leisure and Hospitality
    "80000000",  # Other Services
    "90000000",  # Government
]

industry_subset_df = industry_df[industry_df["industry_code"].isin(industry_keep)].copy()

In [ ]:
state_fips_list = [
    code for code in state_fips_df["state_code"].unique()
    if code != "00"
]

industry_list = industry_subset_df["industry_code"].unique().tolist()
area_list = area_cu_df["area_code"].unique().tolist()

In [ ]:
def build_series(fips_list, ind_list, cpi_list):
    laus_ids = []
    ces_ids = []
    cpi_ids = []
    
    # LAUS 
    for fips in fips_list:
        laus_ids.append(f"LAUST{fips}0000000000003")
    
    # CES 
    for fips in fips_list:
        for ind in ind_list:
            ces_ids.append(f"SMU{fips}00000{ind}11")
            
    # CPI
    for area in cpi_list:
        a = str(area).strip()
        # National and Regions will be monthly
        if a.isdigit():
           cpi_ids.append(f"CUUR{a.zfill(4)}SA0")
        else:
            cpi_ids.append(f"CUUR{a}SA0")
            # Sometime semi-annual
            cpi_ids.append(f"CUUS{a}SA0")
        
    return laus_ids, ces_ids, cpi_ids

laus_ids, ces_ids, cpi_ids = build_series(state_fips_list, industry_list, area_list)

In [65]:
print(len(laus_ids))
print(len(ces_ids))
print(len(cpi_ids))

54
810
102


Now, metrics of interest from 2022-2025 can be extracted using the BLS API with the constructed query strings. I'm using V2 of the API, which requires registration, but has a higher daily request allotment (500 per day). The query ids are batched into chunks to stay under the series per query limit (50).  

In [ ]:
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
headers = {'Content-type': 'application/json'}
all_results = []

def fetch_series(target_list):
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    headers = {'Content-type': 'application/json'}
    results = []
    
    chunks = [target_list[i:i + 50] for i in range(0, len(target_list), 50)]
    
    for chunk in chunks:
        payload = {
            "seriesid": chunk,
            "startyear": "2022",
            "endyear": "2025",
            "registrationkey": API_KEY
        }
        r = requests.post(url, json=payload, headers=headers)
        json_data = r.json()
        
        if json_data.get('status') == 'REQUEST_SUCCEEDED':
            for series in json_data['Results']['series']:
                sid = series['seriesID']
                for item in series['data']:
                    results.append({
                        "series_id": sid,
                        "year": item['year'],
                        "period": item['period'],
                        "value": float(item['value']) if item['value'] != "-" else None
                    })
        time.sleep(1)
        
    return pd.DataFrame(results)

In [67]:
laus_raw = fetch_series(laus_ids)
ces_raw = fetch_series(ces_ids)
cpi_raw = fetch_series(cpi_ids)

In [71]:
print(f"LAUS IDs: {laus_raw[:-2]}")
print(f"CES IDs: {ces_raw[:-2]}")
print(f"CPI IDs: {cpi_raw[:-2]}")

LAUS IDs:                  series_id  year period  value
0     LAUST010000000000003  2025    M12    2.3
1     LAUST010000000000003  2025    M11    2.7
2     LAUST010000000000003  2025    M10    NaN
3     LAUST010000000000003  2025    M09    2.9
4     LAUST010000000000003  2025    M08    2.7
...                    ...   ...    ...    ...
2489  LAUST720000000000003  2022    M07    5.7
2490  LAUST720000000000003  2022    M06    5.8
2491  LAUST720000000000003  2022    M05    5.8
2492  LAUST720000000000003  2022    M04    5.8
2493  LAUST720000000000003  2022    M03    5.8

[2494 rows x 4 columns]
CES IDs:                   series_id  year period    value
0      SMU01000000500000011  2025    M12  1137.78
1      SMU01000000500000011  2025    M11  1152.36
2      SMU01000000500000011  2025    M10  1123.95
3      SMU01000000500000011  2025    M09  1115.77
4      SMU01000000500000011  2025    M08  1115.10
...                     ...   ...    ...      ...
25241  SMU72000000500000011  2022    M07  

In [78]:
def standardize_date(df):
    df = df[df['period'] != 'M13'].copy()
    
    # Map periods to months
    month_map = {
        'M01': '01', 'M02': '02', 'M03': '03', 'M04': '04', 'M05': '05', 'M06': '06',
        'M07': '07', 'M08': '08', 'M09': '09', 'M10': '10', 'M11': '11', 'M12': '12',
        'S01': '06', 'S02': '12'
    }
    df['month'] = df['period'].map(month_map)
    df['date'] = pd.to_datetime(df['year'] + '-' + df['month'] + '-01')
    return df.drop(columns=['month'])

laus_clean = standardize_date(laus_raw)
ces_clean = standardize_date(ces_raw)
cpi_clean = standardize_date(cpi_raw)

In [79]:
laus_clean['state_fips'] = laus_clean['series_id'].str[5:7]
ces_clean['state_fips']  = ces_clean['series_id'].str[3:5]

# Extract industry for industry names too
ces_clean['ind_code'] = ces_clean['series_id'].str[10:18]

In [80]:
laus_clean = laus_clean.merge(state_fips_df[['state_code', 'state_name']], 
                              how = 'left',
                              left_on='state_fips', 
                              right_on='state_code')

ces_clean = ces_clean.merge(state_fips_df[['state_code', 'state_name']], 
                              how = 'left',
                              left_on='state_fips', 
                              right_on='state_code')

ces_clean = ces_clean.merge(industry_df[['industry_code', 'industry_name']], 
                            how='left',
                            left_on='ind_code', 
                            right_on='industry_code')

In [81]:
laus_clean

,series_id,year,period,value,date,state_fips,state_code,state_name
0,LAUST010000000000003,2025,M12,2.3,2025-12-01,01,01,Alabama
1,LAUST010000000000003,2025,M11,2.7,2025-11-01,01,01,Alabama
2,LAUST010000000000003,2025,M10,NaN,2025-10-01,01,01,Alabama
3,LAUST010000000000003,2025,M09,2.9,2025-09-01,01,01,Alabama
4,LAUST010000000000003,2025,M08,2.7,2025-08-01,01,01,Alabama
...,...,...,...,...,...,...,...,...
2491,LAUST720000000000003,2022,M05,5.8,2022-05-01,72,72,Puerto Rico
2492,LAUST720000000000003,2022,M04,5.8,2022-04-01,72,72,Puerto Rico
2493,LAUST720000000000003,2022,M03,5.8,2022-03-01,72,72,Puerto Rico
2494,LAUST720000000000003,2022,M02,6.0,2022-02-01,72,72,Puerto Rico


In [87]:
# Glorified set_index, but for merging
laus_pivot = laus_clean.pivot_table(index=['state_name', 'date'], values='value').rename(columns={'value': 'unemployment_rate'})
ces_pivot = ces_clean.pivot_table(index=['state_name', 'date', 'industry_name'], values='value').rename(columns={'value': 'avg_earnings'})

final_df = ces_pivot.join(laus_pivot, on=['state_name', 'date'], how='left').reset_index()

# National CPI as baseline
nat_cpi = cpi_clean[cpi_clean['series_id'] == 'CUUR0000SA0'][['date', 'value']].rename(columns={'value': 'cpi_national'})
final_df = final_df.merge(nat_cpi, on='date', how='left')

In [88]:
final_df

,state_name,date,industry_name,avg_earnings,unemployment_rate,cpi_national
0,Alabama,2022-01-01,Construction,1086.98,3.1,281.148
1,Alabama,2022-01-01,Financial Activities,1254.52,3.1,281.148
2,Alabama,2022-01-01,Goods Producing,1149.92,3.1,281.148
3,Alabama,2022-01-01,Leisure and Hospitality,388.62,3.1,281.148
4,Alabama,2022-01-01,Manufacturing,1220.14,3.1,281.148
...,...,...,...,...,...,...
25243,Wyoming,2025-12-01,Private Education and Health Services,805.39,3.6,324.054
25244,Wyoming,2025-12-01,Private Service Providing,1000.82,3.6,324.054
25245,Wyoming,2025-12-01,Professional and Business Services,1180.40,3.6,324.054
25246,Wyoming,2025-12-01,Total Private,1147.51,3.6,324.054


Sift through missing values as a final check:

In [ ]:
final_df.isna().sum()

state_name             0
date                   0
industry_name          0
avg_earnings           0
unemployment_rate    525
cpi_national         526
dtype: int64

In [92]:
final_df[final_df['date'] == '2025-10-01']

,state_name,date,industry_name,avg_earnings,unemployment_rate,cpi_national
495,Alabama,2025-10-01,Construction,1377.00,NaN,NaN
496,Alabama,2025-10-01,Financial Activities,1306.52,NaN,NaN
497,Alabama,2025-10-01,Goods Producing,1351.83,NaN,NaN
498,Alabama,2025-10-01,Leisure and Hospitality,445.90,NaN,NaN
499,Alabama,2025-10-01,Manufacturing,1337.74,NaN,NaN
...,...,...,...,...,...,...
25225,Wyoming,2025-10-01,Private Education and Health Services,809.73,NaN,NaN
25226,Wyoming,2025-10-01,Private Service Providing,1009.47,NaN,NaN
25227,Wyoming,2025-10-01,Professional and Business Services,1191.95,NaN,NaN
25228,Wyoming,2025-10-01,Total Private,1153.86,NaN,NaN


Likely by more than coincidence, the extended government shutdown in October 2025 is perfectly overlaid in time by the entire section of NaNs within the final data. It's probably best to estimate these values instead of dropping them.

In [93]:
final_df['cpi_national'] = final_df['cpi_national'].interpolate(method='linear')
# I admit that this one is contestable:
# The BEA reccomends interpolating for Oct but economic theory points to forward/backfilling.
final_df['unemployment_rate'] = final_df.groupby('state_name')['unemployment_rate'].ffill()

In [95]:
final_df[final_df['date'] == '2025-10-01']

,state_name,date,industry_name,avg_earnings,unemployment_rate,cpi_national
495,Alabama,2025-10-01,Construction,1377.00,2.9,324.7435
496,Alabama,2025-10-01,Financial Activities,1306.52,2.9,324.6870
497,Alabama,2025-10-01,Goods Producing,1351.83,2.9,324.6305
498,Alabama,2025-10-01,Leisure and Hospitality,445.90,2.9,324.5740
499,Alabama,2025-10-01,Manufacturing,1337.74,2.9,324.5175
...,...,...,...,...,...,...
25225,Wyoming,2025-10-01,Private Education and Health Services,809.73,2.8,324.4610
25226,Wyoming,2025-10-01,Private Service Providing,1009.47,2.8,324.3932
25227,Wyoming,2025-10-01,Professional and Business Services,1191.95,2.8,324.3254
25228,Wyoming,2025-10-01,Total Private,1153.86,2.8,324.2576


In [99]:
master_df = final_df.rename(columns={'avg_earnings': 'avg_hourly_earnings'})

# Calculate Real Wage Index (Nominal Wage / CPI) * 100
master_df['real_wage_index'] = (master_df['avg_hourly_earnings'] / master_df['cpi_national']) * 100

# 3. Calculate Year-over-Year (YoY) Wage Growth 
master_df = master_df.sort_values(['state_name', 'industry_name', 'date'])
master_df['wage_growth_yoy'] = master_df.groupby(['state_name', 'industry_name'])['avg_hourly_earnings'].pct_change(periods=12)

In [103]:
final_export_df = master_df.dropna(subset=['wage_growth_yoy'])

In [107]:
final_export_df.to_csv("bls_stats.csv")